In [7]:
!pip install torch
!pip install torchvision

In [4]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import matplotlib.pyplot as plt
from PIL import Image

# Cesty k dátam
data_dir = "/home/jovyan/data/lightning/LiviaMurankova/new/namnozene_meteory/DP_rozdelenie_dat/data_split_v2/"
train_dir = os.path.join(data_dir, "train", "images")
valid_dir = os.path.join(data_dir, "valid", "images")
test_dir = os.path.join(data_dir, "test", "images")

train_label_dir = os.path.join(data_dir, "train", "labels")
valid_label_dir = os.path.join(data_dir, "valid", "labels")
test_label_dir = os.path.join(data_dir, "test", "labels")

# Predspracovanie dát
transform = transforms.Compose([
    transforms.Resize((480, 480)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Dataset
class MeteorDataset(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.image_paths = [os.path.join(image_dir, fname) for fname in os.listdir(image_dir) if fname.endswith('.jpg')]
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label_path = os.path.join(self.label_dir, os.path.basename(image_path).replace('.jpg', '.txt'))

        image = Image.open(image_path).convert('RGB')

        if os.path.exists(label_path) and open(label_path).read().strip() == "0":
            label = 1  # Meteor prítomný
        else:
            label = 0  # Meteor neprítomný

        if self.transform:
            image = self.transform(image)

        return image, label

# Načítanie dát
train_dataset = MeteorDataset(train_dir, train_label_dir, transform)
valid_dataset = MeteorDataset(valid_dir, valid_label_dir, transform)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=4, shuffle=False)

# Definícia CNN modelu
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.dropout = nn.Dropout(0.3)

        # Výpočet veľkosti FC vrstvy
        sample_input = torch.randn(1, 3, 480, 480)
        with torch.no_grad():
            feature_size = self._get_feature_map_size(sample_input)

        self.fc1 = nn.Linear(feature_size, 128)
        self.fc2 = nn.Linear(128, 1)
        self.sigmoid = nn.Sigmoid()

    def _get_feature_map_size(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.max_pool2d(x, 2)
        x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2)
        x = torch.relu(self.conv3(x))
        x = torch.max_pool2d(x, 2)
        return x.view(1, -1).size(1)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.max_pool2d(x, 2)
        x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2)
        x = torch.relu(self.conv3(x))
        x = torch.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        x = self.sigmoid(x)
        return x

# Inicializácia
model = CNNModel().cuda()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.000001, weight_decay=1e-4)

# Tréning
num_epochs = 50
train_losses = []
valid_losses = []
train_accuracies = []
valid_accuracies = []

# Early stopping
early_stopping_patience = 5
best_valid_loss = float('inf')
epochs_without_improvement = 0

for epoch in range(num_epochs):
    start_time = time.time()

    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.cuda(), labels.cuda().float()

        optimizer.zero_grad()
        outputs = model(inputs).squeeze(1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        predicted = (outputs > 0.5).float()
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_losses.append(running_loss / len(train_loader))
    train_accuracies.append(correct / total)

    # Validácia
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in valid_loader:
            inputs, labels = inputs.cuda(), labels.cuda().float()
            outputs = model(inputs).squeeze(1)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            predicted = (outputs > 0.5).float()
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    valid_loss = running_loss / len(valid_loader)
    valid_acc = correct / total

    valid_losses.append(valid_loss)
    valid_accuracies.append(valid_acc)

    end_time = time.time()
    epoch_duration = end_time - start_time

    print(f"Epoch {epoch+1}/{num_epochs} - "
          f"Train Loss: {train_losses[-1]:.4f}, Train Accuracy: {train_accuracies[-1]*100:.2f}% - "
          f"Valid Loss: {valid_loss:.4f}, Valid Accuracy: {valid_acc*100:.2f}% - "
          f"Epoch Time: {epoch_duration:.2f}s")

    # Early stopping kontrola
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        epochs_without_improvement = 0
        torch.save(model.state_dict(), 'best_model_early_stopping.pth')
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= early_stopping_patience:
            print(f"Early stopping triggered after {epoch+1} epochs.")
            break

# Grafy
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.plot(train_accuracies, label='Train Accuracy')
plt.plot(valid_accuracies, label='Valid Accuracy')
plt.title('Accuracy during Training')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_losses, label='Train Loss')
plt.plot(valid_losses, label='Valid Loss')
plt.title('Loss during Training')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

# Uloženie finálneho modelu
torch.save(model.state_dict(), 'meteor_detection_cnn_model_v4.pth')

OutOfMemoryError: CUDA out of memory. Tried to allocate 226.00 MiB. GPU 0 has a total capacity of 7.79 GiB of which 87.38 MiB is free. Process 122734 has 6.38 GiB memory in use. Process 169828 has 1.23 GiB memory in use. Process 170178 has 96.00 MiB memory in use. Of the allocated memory 730.00 KiB is allocated by PyTorch, and 1.29 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [17]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

# Nastavenie zariadenia
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Cesty k datasetu
image_dir = "/home/jovyan/data/lightning/LiviaMurankova/new/namnozene_meteory/DP_rozdelenie_dat/data/test/images"
label_dir = "/home/jovyan/data/lightning/LiviaMurankova/new/namnozene_meteory/DP_rozdelenie_dat/data/test/labels"

# Vlastná trieda datasetu
class MeteorDataset(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transform = transform

        # Získanie iba platných súborov obrázkov
        self.image_filenames = sorted([
            fname for fname in os.listdir(image_dir) 
            if fname.endswith('.jpg') and os.path.isfile(os.path.join(image_dir, fname))
        ])

        # Získanie iba platných súborov labelov
        self.label_filenames = sorted([
            fname for fname in os.listdir(label_dir) 
            if fname.endswith('.txt') and os.path.isfile(os.path.join(label_dir, fname))
        ])

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        image_path = os.path.join(self.image_dir, self.image_filenames[idx])
        label_path = os.path.join(self.label_dir, self.image_filenames[idx].replace('.jpg', '.txt'))

        # Načítanie obrázka
        image = Image.open(image_path).convert("RGB")

        # Kontrola, či label existuje a či je súbor platný
        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                label_content = f.read().strip()
            label = 1 if label_content == "0" else 0  # 0 → meteor, prázdny súbor → nie meteor
        else:
            label = 0  # Ak label chýba, považujeme ho za "žiadny meteor"

        if self.transform:
            image = self.transform(image)

        return image, label

# Definovanie transformácií pre dataset
transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
])

# Načítanie testovacieho datasetu
test_dataset = MeteorDataset(image_dir=image_dir, label_dir=label_dir, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Funkcia na evaluáciu modelu
def evaluate_model(model, test_loader, image_filenames):
    model.eval()
    all_preds = []
    all_probs = []
    all_labels = []
    image_names = []

    with torch.no_grad():
        for i, (images, labels) in enumerate(test_loader):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images).squeeze(1)
            probabilities = outputs.cpu().numpy()
            predicted = (outputs > 0.5).cpu().numpy().astype(int)

            all_preds.extend(predicted)
            all_probs.extend(probabilities)
            all_labels.extend(labels.cpu().numpy())

            # Uloženie názvov obrázkov
            batch_filenames = image_filenames[i * test_loader.batch_size: (i + 1) * test_loader.batch_size]
            image_names.extend(batch_filenames)

    # Vytvorenie DataFrame s výsledkami
    df_results = pd.DataFrame({
        "Image Name": image_names,
        "True Label": ["Meteor" if lbl == 1 else "No Meteor" for lbl in all_labels],
        "Predicted Label": ["Meteor" if pred == 1 else "No Meteor" for pred in all_preds],
        "Prediction Probability": all_probs
    })

    # Výpis klasifikačnej správy a konfúznej matice
    print("Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=["No Meteor", "Meteor"], zero_division=1))

    print("Confusion Matrix:")
    print(confusion_matrix(all_labels, all_preds))

    # Výpis časti logov
    print(df_results.head(10))

    # Uloženie výsledkov do CSV
    df_results.to_csv("model_predictions.csv", index=False)
    print("Výsledky boli uložené do 'model_predictions.csv'")

# Načítanie modelu (upravte podľa vašej architektúry)
model = CNNModel()
model.load_state_dict(torch.load("meteor_detection_cnn_model_v2.pth"))
model.to(device)

# Načítanie testovacích názvov obrázkov
test_image_filenames = sorted([
    fname for fname in os.listdir(image_dir) if fname.endswith(".jpg")
])

# Evaluácia modelu
evaluate_model(model, test_loader, test_image_filenames)


Classification Report:
              precision    recall  f1-score   support

   No Meteor       0.08      1.00      0.15        52
      Meteor       1.00      0.00      0.00       594

    accuracy                           0.08       646
   macro avg       0.54      0.50      0.07       646
weighted avg       0.93      0.08      0.01       646

Confusion Matrix:
[[ 52   0]
 [594   0]]
                  Image Name True Label Predicted Label  \
0  M20210708_220210_FO_P.jpg     Meteor       No Meteor   
1  M20210708_235512_FO_P.jpg     Meteor       No Meteor   
2  M20210708_235536_FO_P.jpg     Meteor       No Meteor   
3  M20210709_000301_FO_P.jpg     Meteor       No Meteor   
4  M20210709_000456_FO_P.jpg     Meteor       No Meteor   
5  M20210709_005352_FO_P.jpg     Meteor       No Meteor   
6  M20210709_005810_FO_P.jpg     Meteor       No Meteor   
7  M20210709_205427_FO_P.jpg     Meteor       No Meteor   
8  M20210709_213808_FO_P.jpg     Meteor       No Meteor   
9  M20210709_222934